In [0]:
#TESTING - NOT NECESSARY TO RUN

from pyspark.sql import functions as F

file250_path = "/Volumes/workspace/default/sephora_surfers/reviews_0-250.csv"

CHECK250 = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)   # keep everything as string first
    .option("multiLine", True)      # safer for review text
    .option("quote", '"')
    .option("escape", '"')
    .csv(file250_path)
)

print("Total rows:", CHECK250.count())
display(CHECK250)

In [0]:
#NOT NECESSARY TO RUN. THERE IS NO NULL FOR EACH FILE. 
for file in reviews_files:
    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", False)
        .option("multiLine", True)
        .option("quote", '"')
        .option("escape", '"')
        .csv(base_path + file)
    )

    null_count = df.filter(F.col("product_id").isNull()).count()
    print(file, "null product_id rows:", null_count)

In [0]:
#IGNORE. THIS HAS NO CONSISTENT SCHEMA INFERENCE
from pyspark.sql import functions as F

base_path = "/Volumes/workspace/default/sephora_surfers/"

reviews_files = [
    "reviews_0-250.csv",
    "reviews_250-500.csv",
    "reviews_500-750.csv",
    "reviews_750-1250.csv",
    "reviews_1250-end.csv"
]

# Load and union all review datasets
reviews_df = None

for file in reviews_files:
    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(base_path + file)
    )
    
    if reviews_df is None:
        reviews_df = df
    else:
        reviews_df = reviews_df.unionByName(df)

print("Total rows:", reviews_df.count())
display(reviews_df)

In [0]:
#START FROM HERE. CONSISTENT SCHEMA INFERENCE
from pyspark.sql import functions as F

base_path = "/Volumes/workspace/default/sephora_surfers/"

reviews_files = [
    "reviews_0-250.csv",
    "reviews_250-500.csv",
    "reviews_500-750.csv",
    "reviews_750-1250.csv",
    "reviews_1250-end.csv"
]

reviews_df = None

for file in reviews_files:
    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", False)   # keep everything as string first
        .option("multiLine", True)      # important for review text
        .option("quote", '"')
        .option("escape", '"')
        .option("mode", "PERMISSIVE")
        .csv(base_path + file)
    )

    # normalize the unnamed first column if present
    if "Unnamed: 0" in df.columns:
        df = df.withColumnRenamed("Unnamed: 0", "row_id")
    elif "_c0" in df.columns:
        df = df.withColumnRenamed("_c0", "row_id")
    elif "c0" in df.columns:
        df = df.withColumnRenamed("c0", "row_id")

    if reviews_df is None:
        reviews_df = df
    else:
        reviews_df = reviews_df.unionByName(df, allowMissingColumns=True)

print("Total rows:", reviews_df.count())
display(reviews_df)

In [0]:
def spark_shape(df):
    return (df.count(), len(df.columns))

spark_shape(reviews_df)

In [0]:
#CHECK PRODUCT INFO
product_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(base_path + "product_info.csv")
)

display(product_df)

In [0]:
#JOIN PRODUCT INFO WITH RATINGS & REVIEWS FILE
reviews_enriched = reviews_df.join(
    product_df,
    on="product_id",
    how="left"
)

display(reviews_enriched)

In [0]:
def spark_shape(df):
    return (df.count(), len(df.columns))

spark_shape(reviews_enriched)

In [0]:
#product_df_small = product_df.select(
#    "product_id",
#    "product_name",
#    "loves_count"
#).withColumnRenamed("product_name", "product_name_info")
product_df_small = product_df.select(
    "product_id",
    "product_name",
    "ingredients",
    "loves_count"
).withColumnRenamed(
    "product_name",
    "product_name_info"
)

In [0]:
#JOIN PRODUCT WITH RATINGS REVIEWS FILE
reviews_enriched = reviews_df.join(
    product_df_small,
    on="product_id",
    how="left"
)

In [0]:
#SAVE DELTA TABLE TO WORKSPANCE. THIS IS TABLE WE WILL USE FOR MODELLING
#TAKE INTO ACCOUNT THAT THE TABLE NAME SAVED AS DELTA TABLE BELOW MAY CHANGE DEPENDING ON WHETHER WE WANT TO CREATE SEPARATE TABLE FOR SEPARATE MODELS
reviews_enriched.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.sephora_reviews_enrichedSN")

In [0]:
%sql
SELECT *
FROM workspace.default.sephora_reviews_enrichedSN;

In [0]:
#COUNT ROWS & COLUMNS
spark_shape(reviews_enriched)

In [0]:
#COUNT NUMER OF ROWS WITH RATINGS
from pyspark.sql import functions as F

df = spark.table("workspace.default.sephora_reviews_enrichedSN")

result = df.select(
    F.sum(F.when(F.col("rating") >= 4, 1).otherwise(0)).alias("relevant_rows"),
    F.count("*").alias("total_rows")
)

result.show()

In [0]:
from pyspark.sql import functions as F

df = spark.table("workspace.default.sephora_reviews_enrichedSN")

ratings_1_5 = spark.createDataFrame([(1,),(2,),(3,),(4,),(5,)], ["rating"])

rating_dist = (
    ratings_1_5
    .join(df.groupBy("rating").count(), on="rating", how="left")
    .fillna(0)
    .withColumn(
        "percentage",
        F.round(100 * F.col("count") / df.count(), 2)
    )
    .orderBy("rating")
)

display(rating_dist)

In [0]:
from pyspark.sql import functions as F

df = spark.table("workspace.default.sephora_reviews_enrichedSN")

is_recommended_dist = (
    df.groupBy("is_recommended")
      .agg(F.count("*").alias("count"))
      .withColumn("percentage", F.round(100 * F.col("count") / df.count(), 2))
      .orderBy("is_recommended")
)

display(is_recommended_dist)

In [0]:
from pyspark.sql import functions as F

df = spark.table("workspace.default.sephora_reviews_enrichedSN")

total_rating5 = df.filter(F.col("rating") == 5).count()
rating5_recommended1 = df.filter((F.col("rating") == 5) & (F.col("is_recommended").cast("double") == 1.0)).count()

percentage = (rating5_recommended1 / total_rating5) * 100 if total_rating5 > 0 else None

display(spark.createDataFrame(
    [(total_rating5, rating5_recommended1, percentage)],
    ["total_rating5", "rating5_recommended1", "percentage"]
))

In [0]:
from pyspark.sql import functions as F

df = spark.table("workspace.default.sephora_reviews_enrichedSN")

total_rows = df.count()

rating_extra = (
    df
    .groupBy("rating")
    .agg(
        F.count("*").alias("count"),

        F.sum(
            F.when(
                (F.col("rating") == 5) &
                (F.col("is_recommended").cast("double") == 1),
                1
            ).otherwise(0)
        ).alias("rating5_recommended1")
    )

    .withColumn(
        "percentage",
        F.col("count") / total_rows * 100
    )

    .withColumn(
        "pct_rating5_recommended1",
        F.col("rating5_recommended1") / F.col("count") * 100
    )

    .orderBy("rating")
)

rating_extra.show()

In [0]:
df = spark.table("workspace.default.sephora_reviews_enrichedSN")
display(df.limit(10))

In [0]:
rating_extra = (
    df
    .groupBy("rating")
    .agg(
        F.count("*").alias("count"),
        F.sum(
            F.when(
                (F.col("rating") == 5) & (F.col("is_recommended") == 1),
                1
            ).otherwise(0)
        ).alias("rating5_recommended1")
    )
    .withColumn(
        "percentage",
        F.col("count") / df.count() * 100
    )
    .withColumn(
        "pct_rating5_recommended1",
        F.col("rating5_recommended1") / F.col("count") * 100
    )
    .orderBy("rating")
)

rating_extra.show()

In [0]:
#CHECK FOR ERRORS
from pyspark.sql import functions as F

df = spark.table("workspace.default.sephora_reviews_enrichedSN")

df2 = (
    df.withColumn("product_id_trim", F.trim(F.col("product_id").cast("string")))
      .withColumn("author_id_trim", F.trim(F.col("author_id").cast("string")))
      .withColumn("rating_trim", F.trim(F.col("rating").cast("string")))
      .withColumn("review_text_trim", F.trim(F.col("review_text").cast("string")))
      .withColumn("review_title_trim", F.trim(F.col("review_title").cast("string")))
)

unclean_df = (
    df2
    .withColumn("bad_product_id",
                F.col("product_id").isNull() | (F.col("product_id_trim") == ""))
    .withColumn("bad_author_id",
                F.col("author_id").isNull() | (F.col("author_id_trim") == ""))
    .withColumn("bad_rating_non_numeric",
                F.col("rating").isNotNull() & F.expr("try_cast(rating as double) is null"))
    .withColumn("bad_rating_out_of_range",
                F.expr("try_cast(rating as double) is not null") &
                ((F.expr("try_cast(rating as double)") < 1) | (F.expr("try_cast(rating as double)") > 5)))
    .withColumn("bad_review_text_blank",
                F.col("review_text").isNull() | (F.col("review_text_trim") == ""))
    .withColumn("bad_is_recommended",
                F.col("is_recommended").isNotNull() &
                (~F.col("is_recommended").cast("string").isin("0", "1", "0.0", "1.0", "true", "false", "True", "False")))
    
    # numeric quality checks using try_cast
    .withColumn("bad_total_feedback_non_numeric",
                F.col("total_feedback_count").isNotNull() &
                F.expr("try_cast(total_feedback_count as double) is null"))
    .withColumn("bad_total_feedback_count",
                F.expr("try_cast(total_feedback_count as double) is not null") &
                (F.expr("try_cast(total_feedback_count as double)") < 0))

    .withColumn("bad_total_pos_feedback_non_numeric",
                F.col("total_pos_feedback_count").isNotNull() &
                F.expr("try_cast(total_pos_feedback_count as double) is null"))
    .withColumn("bad_total_pos_feedback_count",
                F.expr("try_cast(total_pos_feedback_count as double) is not null") &
                (F.expr("try_cast(total_pos_feedback_count as double)") < 0))

    .withColumn("bad_total_neg_feedback_non_numeric",
                F.col("total_neg_feedback_count").isNotNull() &
                F.expr("try_cast(total_neg_feedback_count as double) is null"))
    .withColumn("bad_total_neg_feedback_count",
                F.expr("try_cast(total_neg_feedback_count as double) is not null") &
                (F.expr("try_cast(total_neg_feedback_count as double)") < 0))

    .withColumn("bad_helpfulness_non_numeric",
                F.col("helpfulness").isNotNull() &
                F.expr("try_cast(helpfulness as double) is null"))

    .withColumn("has_any_issue",
                F.col("bad_product_id") |
                F.col("bad_author_id") |
                F.col("bad_rating_non_numeric") |
                F.col("bad_rating_out_of_range") |
                F.col("bad_review_text_blank") |
                F.col("bad_is_recommended") |
                F.col("bad_total_feedback_non_numeric") |
                F.col("bad_total_feedback_count") |
                F.col("bad_total_pos_feedback_non_numeric") |
                F.col("bad_total_pos_feedback_count") |
                F.col("bad_total_neg_feedback_non_numeric") |
                F.col("bad_total_neg_feedback_count") |
                F.col("bad_helpfulness_non_numeric")
               )
    .filter(F.col("has_any_issue"))
)

display(unclean_df)

In [0]:
#SUMMARY OF ERRORS. CONCLUSION - NO NEED TO DELETE ANY COLUMNS. ONLY HAVE MISSING TEXT REVIEWS WHICH IS NORMAL
summary_df = unclean_df.select(
    F.sum(F.col("bad_product_id").cast("int")).alias("bad_product_id"),
    F.sum(F.col("bad_author_id").cast("int")).alias("bad_author_id"),
    F.sum(F.col("bad_rating_non_numeric").cast("int")).alias("bad_rating_non_numeric"),
    F.sum(F.col("bad_rating_out_of_range").cast("int")).alias("bad_rating_out_of_range"),
    F.sum(F.col("bad_review_text_blank").cast("int")).alias("bad_review_text_blank"),
    F.sum(F.col("bad_is_recommended").cast("int")).alias("bad_is_recommended"),
    F.sum(F.col("bad_total_feedback_count").cast("int")).alias("bad_total_feedback_count"),
    F.sum(F.col("bad_total_pos_feedback_count").cast("int")).alias("bad_total_pos_feedback_count"),
    F.sum(F.col("bad_total_neg_feedback_count").cast("int")).alias("bad_total_neg_feedback_count"),
    F.sum(F.col("bad_helpfulness_non_numeric").cast("int")).alias("bad_helpfulness_non_numeric")
)

display(summary_df)

In [0]:
unclean_df.count()

In [0]:
#CHECK FOR ERRORS - Added check null values for numeric values (to check if need to replace cell 14)
from pyspark.sql import functions as F

df2 = (
    df_enriched.withColumn("product_id_trim", F.trim(F.col("product_id").cast("string")))
      .withColumn("row_id_trim", F.trim(F.col("row_id").cast("string")))
      .withColumn("author_id_trim", F.trim(F.col("author_id").cast("string")))
      .withColumn("rating_trim", F.trim(F.col("rating").cast("string")))
      .withColumn("review_text_trim", F.trim(F.col("review_text").cast("string")))
      .withColumn("review_title_trim", F.trim(F.col("review_title").cast("string")))
)

unclean_df = (
    df2
    .withColumn("bad_product_id",
                F.col("product_id").isNull() | (F.col("product_id_trim") == ""))
    .withColumn("bad_row_id",
                F.col("row_id").isNull() | (F.col("row_id_trim") == ""))
    .withColumn("bad_author_id",
                F.col("author_id").isNull() | (F.col("author_id_trim") == ""))
    .withColumn("bad_rating_non_numeric",
                F.col("rating").isNotNull() & F.expr("try_cast(rating as double) is null"))
    .withColumn("bad_rating_out_of_range",
                F.expr("try_cast(rating as double) is not null") &
                ((F.expr("try_cast(rating as double)") < 1) | (F.expr("try_cast(rating as double)") > 5)))
    .withColumn("bad_review_text_blank",
                F.col("review_text").isNull() | (F.col("review_text_trim") == ""))
    .withColumn("bad_is_recommended",
                F.col("is_recommended").isNotNull() &
                (~F.col("is_recommended").cast("string").isin("0", "1", "0.0", "1.0", "true", "false", "True", "False")))
    
    # numeric quality checks using try_cast
    .withColumn("bad_total_feedback_non_numeric",
                F.col("total_feedback_count").isNotNull() &
                F.expr("try_cast(total_feedback_count as double) is null"))
    .withColumn("bad_total_feedback_count",
                F.col("total_feedback_count").isNull() | (F.expr("try_cast(total_feedback_count as double) is not null") &
                (F.expr("try_cast(total_feedback_count as double)") < 0)))

    .withColumn("bad_total_pos_feedback_non_numeric",
                F.col("total_pos_feedback_count").isNotNull() &
                F.expr("try_cast(total_pos_feedback_count as double) is null"))
    .withColumn("bad_total_pos_feedback_count",
                F.col("total_pos_feedback_count").isNull() | (F.expr("try_cast(total_pos_feedback_count as double) is not null") &
                (F.expr("try_cast(total_pos_feedback_count as double)") < 0)))

    .withColumn("bad_total_neg_feedback_non_numeric",
                F.col("total_neg_feedback_count").isNotNull() &
                F.expr("try_cast(total_neg_feedback_count as double) is null"))
    .withColumn("bad_total_neg_feedback_count",
                F.col("total_neg_feedback_count").isNull() | (F.expr("try_cast(total_neg_feedback_count as double) is not null") &
                (F.expr("try_cast(total_neg_feedback_count as double)") < 0)))

    .withColumn("bad_helpfulness_non_numeric",
                F.col("helpfulness").isNull() | (F.col("helpfulness").isNotNull() &
                F.expr("try_cast(helpfulness as double) is null")))

    .withColumn("has_any_issue",
                F.col("bad_product_id") |
                F.col("bad_row_id") |
                F.col("bad_author_id") |
                F.col("bad_rating_non_numeric") |
                F.col("bad_rating_out_of_range") |
                F.col("bad_review_text_blank") |
                F.col("bad_is_recommended") |
                F.col("bad_total_feedback_non_numeric") |
                F.col("bad_total_feedback_count") |
                F.col("bad_total_pos_feedback_non_numeric") |
                F.col("bad_total_pos_feedback_count") |
                F.col("bad_total_neg_feedback_non_numeric") |
                F.col("bad_total_neg_feedback_count") |
                F.col("bad_helpfulness_non_numeric")
               )
    .filter(F.col("has_any_issue"))
)

display(unclean_df)

In [0]:
ratings.groupBy("rating").count().orderBy("rating").display()

In [0]:
from pyspark.sql import functions as F

# Recreate the full join temporarily (don't save it)
reviews_enriched_full = reviews_df.join(
    product_df,
    on="product_id",
    how="left"
)

print(f"Total columns: {len(reviews_enriched_full.columns)}")
print("\n=== ALL 45 COLUMNS ===")
for i, col in enumerate(reviews_enriched_full.columns, 1):
    print(f"{i}. {col}")

print("\n=== SCHEMA DETAILS ===")
reviews_enriched_full.printSchema()